RUNTIME: CPU (no GPU needed for this notebook)    
INSTRUCTIONS:

1. Set runtime to CPU: Runtime → Change runtime type → None
2. Run all cells in order: Runtime → Run all
3. When prompted, authorize Google Drive access
4. Model checkpoints are saved to Google Drive via:
```
    import joblib
    joblib.dump(model, SAVE_DIR + "checkpoints/svm_model.joblib")
```
Make sure SAVE_DIR is set correctly in the setup cell    

5. To save results/figures to GitHub:    
    a. Right-click the file in the left sidebar → Download    
    b. Go to the GitHub repo → results/ → Add file → Upload files    
    c. Commit directly on GitHub
6. Save the notebook: Ctrl+S → save to GitHub when prompted

In [30]:
# ── 1. Standard imports ──────────────────────────────────────────
import ast
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC

# Resolve repo root so src imports work in local Jupyter and Colab
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src" / "evaluate.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo root containing src/evaluate.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_run

print("Repo root:", repo_root)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
# ── 2. Data loading helpers ──────────────────────────────────────
def _resolve_data_dir():
    for candidate in [repo_root / "data", Path("../data"), Path("data")]:
        if (candidate / "train.csv").exists() and (candidate / "test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory with train.csv/test.csv")

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()

    def _parse_one(value):
        if isinstance(value, list):
            return [int(x) for x in value]
        if isinstance(value, np.ndarray):
            return [int(x) for x in value.tolist()]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if body == "":
                    return []
                if ',' in body:
                    toks = [t.strip() for t in body.split(',') if t.strip()]
                else:
                    toks = [t for t in body.split() if t]
                return [int(t) for t in toks]
            return [int(x) for x in ast.literal_eval(s)]
        return value

    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _choose_text_col(df):
    for col in ["text_clean_tfidf", "text"]:
        if col in df.columns:
            return col
    raise ValueError("No text column found. Expected one of: text_clean_tfidf, text")

def _get_label_names(train_df):
    # Prefer canonical label names from HF feature schema.
    try:
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
        return ds["train"].features["labels"].feature.names
    except Exception:
        max_id = max(max(labels) if labels else -1 for labels in train_df["labels"])
        return [f"label_{i}" for i in range(max_id + 1)]

data_dir = _resolve_data_dir()
results_dir = repo_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("Data dir:", data_dir)
print("Results dir:", results_dir)


Saved test.csv to /content/drive/MyDrive/cs4120_project/data/test.csv
Saved train.csv to /content/drive/MyDrive/cs4120_project/data/train.csv
Saved data/train_1pct.csv (1.0%) to /content/drive/MyDrive/cs4120_project/data/train_1pct.csv
Saved data/train_5pct.csv (5.0%) to /content/drive/MyDrive/cs4120_project/data/train_5pct.csv
Saved data/train_10pct.csv (10.0%) to /content/drive/MyDrive/cs4120_project/data/train_10pct.csv
Saved data/train_25pct.csv (25.0%) to /content/drive/MyDrive/cs4120_project/data/train_25pct.csv
Saved data/train_50pct.csv (50.0%) to /content/drive/MyDrive/cs4120_project/data/train_50pct.csv
All data files generated and saved to Google Drive!


In [32]:
# ── 3. Experiment configuration ──────────────────────────────────
METHOD_NAME = "svm_tfidf"
DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 7, 21]

TFIDF_CONFIG = {
    "max_features": 10000,
    "ngram_range": (1, 2),
}
SVM_CONFIG = {
    "class_weight": "balanced",
}

def _fraction_to_pct_tag(frac):
    return f"{int(round(float(frac) * 100))}pct"

def _train_file_for_fraction_seed(frac, seed):
    frac = float(frac)
    if frac == 1.0:
        return data_dir / "train.csv"

    pct_tag = _fraction_to_pct_tag(frac)
    seeded = data_dir / f"train_{pct_tag}_seed{seed}.csv"
    canonical = data_dir / f"train_{pct_tag}.csv"

    if seeded.exists():
        return seeded
    if canonical.exists():
        return canonical

    raise FileNotFoundError(f"No train file found for frac={frac}, seed={seed}")

print("Configured fractions:", DATA_FRACTIONS)
print("Configured seeds:", SEEDS)


In [33]:
# ── 4. Training + Evaluation loop (shared framework) ────────────
def _set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

test_df = _parse_labels_column(pd.read_csv(data_dir / "test.csv"), label_col="labels")
text_col = _choose_text_col(test_df)

# Use full train split to infer label space robustly
train_full_df = _parse_labels_column(pd.read_csv(data_dir / "train.csv"), label_col="labels")
label_names = _get_label_names(train_full_df)

mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
mlb.fit([list(range(len(label_names)))])
y_test = mlb.transform(test_df["labels"])

all_overall = []
all_per_class = []
run_errors = []

checkpoints_dir = results_dir / "checkpoints" / "svm_tfidf"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    for frac in DATA_FRACTIONS:
        print(f"\n=== SVM run: fraction={frac}, seed={seed} ===")
        try:
            _set_seed(seed)
            train_path = _train_file_for_fraction_seed(frac, seed)
            train_df = _parse_labels_column(pd.read_csv(train_path), label_col="labels")

            if text_col not in train_df.columns:
                text_col = _choose_text_col(train_df)

            y_train = mlb.transform(train_df["labels"])

            vectorizer = TfidfVectorizer(
                max_features=TFIDF_CONFIG["max_features"],
                ngram_range=TFIDF_CONFIG["ngram_range"],
            )

            train_text = train_df[text_col].fillna("").astype(str)
            test_text = test_df[text_col].fillna("").astype(str)

            X_train = vectorizer.fit_transform(train_text)
            X_test = vectorizer.transform(test_text)

            svm = OneVsRestClassifier(
                LinearSVC(random_state=int(seed), class_weight=SVM_CONFIG["class_weight"])
            )
            svm.fit(X_train, y_train)

            if float(frac) == 1.0:
                joblib.dump(
                    (vectorizer, svm),
                    checkpoints_dir / f"svm_model_seed{seed}.joblib",
                )

            y_pred = svm.predict(X_test)

            evaluation = evaluate_run(
                method=METHOD_NAME,
                data_fraction=float(frac),
                seed=int(seed),
                label_names=label_names,
                y_true=y_test,
                y_pred=y_pred,
            )

            all_overall.append(evaluation["overall"])
            all_per_class.append(evaluation["per_class"])

            macro_f1 = float(evaluation["overall"].iloc[0]["macro_f1"])
            print(f"Completed: macro_f1={macro_f1:.4f}")

        except Exception as exc:
            run_errors.append({"seed": seed, "data_fraction": frac, "error": str(exc)})
            print(f"FAILED: {exc}")

print("\nTraining loop complete.")


Training on 1.0% of data (data/train_1pct.csv)...
Training on 5.0% of data (data/train_5pct.csv)...
Training on 10.0% of data (data/train_10pct.csv)...
Training on 25.0% of data (data/train_25pct.csv)...
Training on 50.0% of data (data/train_50pct.csv)...
Training on 100.0% of data (data/train.csv)...
Saved full model to Google Drive!
Experimentation complete!


In [34]:
# ── 5. Save results ──────────────────────────────────────────────
if all_overall:
    overall_df = pd.concat(all_overall, ignore_index=True).sort_values(["seed", "data_fraction"])
    per_class_df = pd.concat(all_per_class, ignore_index=True).sort_values(["seed", "data_fraction", "emotion"])

    overall_path = results_dir / "svm_tfidf_overall.csv"
    per_class_path = results_dir / "svm_tfidf_per_class.csv"
    readme_path = results_dir / "svm_tfidf_results.csv"

    overall_df.to_csv(overall_path, index=False)
    per_class_df.to_csv(per_class_path, index=False)

    # README-compatible long format
    readme_df = per_class_df[["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
    readme_df.to_csv(readme_path, index=False)

    print("Saved outputs:")
    print(overall_path)
    print(per_class_path)
    print(readme_path)

    display(overall_df.head())
    display(per_class_df.head())
else:
    print("No successful runs; no result files were saved.")

if run_errors:
    print("\nRun errors:")
    display(pd.DataFrame(run_errors))
else:
    print("\nAll runs completed without exceptions.")


,method,data_fraction,seed,emotion,f1,precision,recall
0,svm_tfidf,0.01,42,admiration,0.170543,0.390071,0.109127
1,svm_tfidf,0.01,42,amusement,0.370588,0.828947,0.238636
2,svm_tfidf,0.01,42,anger,0.063636,0.318182,0.035354
3,svm_tfidf,0.01,42,annoyance,0.023529,0.200000,0.012500
4,svm_tfidf,0.01,42,approval,0.016260,0.166667,0.008547
5,svm_tfidf,0.01,42,caring,0.049383,0.148148,0.029630
6,svm_tfidf,0.01,42,confusion,0.012821,0.333333,0.006536
7,svm_tfidf,0.01,42,curiosity,0.039735,0.333333,0.021127
8,svm_tfidf,0.01,42,desire,0.023810,1.000000,0.012048
9,svm_tfidf,0.01,42,disappointment,0.000000,0.000000,0.000000


Results saved to results/svm_tfidf_results.csv.
